# Benchmark de preprocesado OCR - Manualito

Este notebook compara técnicas de preprocesado de imagen para decidir qué transformación conviene aplicar antes del OCR en Manualito.

Se evalúan los tres motores que nos interesan en el proyecto:

- `tesseract`
- `paddle_cpu`
- `paddle_gpu`

La salida queda guardada en `outputs/results.json`, `outputs/results.md` y `outputs/conclusions.md`, de forma que el benchmark puede ejecutarse por partes sin repetir motores lentos.


## 1. Objetivo

Manualito recibe fotos o documentos de manuales de juegos de mesa. El OCR no trabaja con escaneos perfectos, sino con capturas reales: perspectiva, sombras, texto pequeño, columnas, listas, iconos y tipografías variadas.

Por eso este benchmark no busca una técnica universal. Busca una técnica razonable para nuestro caso de uso:

1. Mejorar la lectura frente a la imagen original.
2. Mantener un coste de preprocesado bajo.
3. Evitar conclusiones basadas en una sola imagen fácil.
4. Dejar resultados reproducibles para la memoria del TFG.


## 2. Qué imágenes usar

La literatura y la documentación de herramientas OCR apuntan en la misma dirección: un benchmark útil necesita imágenes con ground truth y una muestra representativa del uso real, no solo capturas limpias.

| Criterio                | Aplicación en Manualito                                                                               |
| ----------------------- | ----------------------------------------------------------------------------------------------------- |
| Ground truth revisado   | Cada imagen debe tener texto de referencia escrito o revisado a mano. Sin esto no hay CER/WER fiable. |
| Escenarios reales       | Priorizar fotos propias de manuales de juegos de mesa, no documentos genéricos.                       |
| Dificultad variada      | Mezclar capturas limpias, perspectiva, sombras, baja luz, desenfoque leve, columnas y texto pequeño.  |
| Español                 | Incluir tildes, `ñ`, signos de apertura y vocabulario típico de reglas.                               |
| Estructura de manual    | Probar listas numeradas, tablas, ejemplos, encabezados, notas y texto junto a iconos.                 |
| Separación de conjuntos | Usar unas imágenes para ajustar la técnica y otras para confirmar que no hemos sobreajustado.         |

El dataset actual combina imágenes libres reales con controles sintéticos generados de forma reproducible. Las imágenes sintéticas no sustituyen a fotos reales, pero sirven para tener casos controlados de texto limpio, sombra y perspectiva.

Para una conclusión final fuerte conviene ampliarlo a 12 imágenes propias de Manualito, separando 8 para ajuste y 4 para validación:

| Tipo recomendado                           |   Ajuste | Validación | Motivo                                            |
| ------------------------------------------ | -------: | ---------: | ------------------------------------------------- |
| Foto frontal limpia de manual de juego     |        2 |          1 | Caso feliz de producción.                         |
| Foto con perspectiva o curvatura de página |        2 |          1 | Captura normal con móvil.                         |
| Foto con sombra, reflejo o baja luz        |        2 |          1 | Estrés realista de mesa/casa.                     |
| Texto pequeño, columnas, listas o tablas   |        2 |          1 | Donde más se castiga el OCR.                      |
| PDF o página escaneada                     | opcional |   opcional | útil para la tarea de varias páginas/PDF.         |
| Imagen histórica o muy degradada           | opcional |   opcional | Caso de estrés, pero no debe dominar la decisión. |

Reglas prácticas para ampliarlo:

- Añadir solo 1-2 imágenes por caso de prueba; muchas páginas casi iguales inflan el benchmark sin aportar señal.
- Revisar el ground truth a mano, idealmente dos veces o con una segunda lectura de control.
- Guardar en `manifest.json` el split, dificultad, condiciones de captura, origen y estado de derechos.
- No mezclar imágenes usadas para elegir parámetros con imágenes usadas para confirmar la recomendación final.
- Evitar subir al repo páginas completas de manuales comerciales si no hay permiso o licencia clara.

Las imágenes históricas o de prensa son útiles para estrés técnico, pero la recomendación final debe salir de manuales reales de juegos de mesa. Si una técnica gana solo en documentos antiguos y pierde en manuales modernos, no debería activarse por defecto.


## 3. Metodología

- Se compara cada técnica contra `baseline`, que es la imagen original sin preprocesado.
- La comparación se hace por separado para cada motor: cada uno tiene su baseline, sus deltas y su recomendación.
- La calidad se mide con CER y WER frente a fragmentos de ground truth.
- El warm-up de cada motor queda fuera de la medición.
- Se mide por separado el tiempo de preprocesado y el tiempo de OCR.
- Los resultados se guardan por motor, pero los tres motores pueden ejecutarse en una sola pasada si el entorno tiene Paddle GPU y Tesseract.
- Una técnica solo se recomienda si mejora el CER medio y mejora al menos la mitad de las imágenes.

Referencias usadas:

- [MathWorks evaluateOCR](https://www.mathworks.com/help/vision/ref/evaluateocr.html): evaluación OCR contra ground truth con CER/WER.
- [PaddleOCR benchmark](https://www.paddleocr.ai/v2.10.0/en/infer_deploy/benchmark.html): benchmark con imágenes de escenarios reales y medición completa del pipeline.
- [PaddleOCR datasets](https://www.paddleocr.ai/main/en/datasets/ocr_datasets.html): organizaci?n de imágenes y anotaciones de ground truth.
- [SmartDoc-QA / document image quality](https://www.frontiersin.org/journals/signal-processing/articles/10.3389/frsip.2026.1779355/full): capturas desde ?ngulos distintos con blur, iluminaci?n, sombras y distorsi?n geom?trica.
- [NIST OCR evaluation](https://tsapps.nist.gov/publication/get_pdf.cfm?pub_id=151348): clasificaci?n de calidad de imagen y ground truth manual verificado.
- [Google Benchmark User Guide](https://github.com/google/benchmark/blob/main/docs/user_guide.md): warm-up y separaci?n de tiempos medidos.


## 4. Configuración

`RUN_BENCHMARK` está a `False` para que abrir el notebook no lance los motores OCR por accidente. Para ejecutar una prueba real, cambia `RUN_BENCHMARK = True`.

`ENGINES = ["tesseract", "paddle_cpu", "paddle_gpu"]` ejecuta los tres motores en la misma pasada. Para eso usa el entorno de Paddle GPU: `paddle_cpu` fuerza `device="cpu"` sobre el mismo paquete de Paddle, y `paddle_gpu` fuerza `device="gpu"`.

`ALLOW_ENGINE_SKIP = False` hace que el benchmark sea estricto: si pides tres motores y uno falla, la ejecución falla. Si alguna vez quieres una pasada parcial para depurar, cambia ese valor a `True`.

`TECHNIQUE_PRESET = "compact"` prueba 28 configuraciones. `TECHNIQUE_PRESET = "extended"` sube a 60 para afinar parámetros por motor.

`INCLUDE_SYNTHETIC_IMAGES = True` añade imágenes sintéticas reproducibles al dataset antes de cargar los casos.


In [1]:
from __future__ import annotations

import contextlib
import hashlib
import importlib
import io
import json
import logging
import os
import re
import statistics
import sys
import time
import warnings
from collections import defaultdict
from collections.abc import Callable
from datetime import UTC, datetime
from functools import partial
from itertools import product
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw, ImageFilter, ImageFont

os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
os.environ.setdefault("FLAGS_log_level", "3")
os.environ.setdefault("GLOG_minloglevel", "2")
os.environ.setdefault("KMP_WARNINGS", "0")

cv2 = importlib.import_module("cv2")
np = importlib.import_module("numpy")

warnings.filterwarnings("ignore")
logging.getLogger("ppocr").setLevel(logging.ERROR)
logging.getLogger("paddle").setLevel(logging.ERROR)

RUN_BENCHMARK = True
FORCE = False
ALLOW_ENGINE_SKIP = False
INCLUDE_SYNTHETIC_IMAGES = True
FORCE_SYNTHETIC_IMAGES = False
TECHNIQUE_PRESET = "extended"  # "compact" = 28 configs, "extended" = 60 configs
ENGINES = ["tesseract", "paddle_cpu", "paddle_gpu"]
REPEATS = 1
TOP_N = 8
DETECTED_WER_THRESHOLD = 0.35
NOTEBOOK_VERSION = "2026-06-04"

ESC = chr(27)
YELLOW = f"{ESC}[33m"
CYAN = f"{ESC}[36m"
GREEN = f"{ESC}[32m"
RED = f"{ESC}[31m"
RESET = f"{ESC}[0m"


def info(msg: str, data: str = "") -> None:
    suffix = f" {data}" if data else ""
    print(f"{YELLOW}[*]{RESET} {CYAN}{msg}{RESET}{suffix}")


def ok(msg: str, data: str = "") -> None:
    suffix = f" {data}" if data else ""
    print(f"{GREEN}[+]{RESET} {msg}{suffix}")


def warn(msg: str) -> None:
    print(f"{YELLOW}[!]{RESET} {msg}")


def fail(msg: str) -> None:
    print(f"{RED}[x] {msg}{RESET}")


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró pyproject.toml")


def resolve_paths() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    if (current / "dataset" / "manifest.json").exists():
        bench_dir = current
        repo_root = find_repo_root(current)
    else:
        repo_root = find_repo_root(current)
        bench_dir = repo_root / "docs" / "benchmarks" / "ocr_preprocessing"
    return repo_root, bench_dir


REPO_ROOT, BENCH_DIR = resolve_paths()
DATASET_DIR = BENCH_DIR / "dataset"
GROUND_TRUTH_DIR = DATASET_DIR / "ground_truth"
OUTPUT_DIR = BENCH_DIR / "outputs"
TMP_DIR = OUTPUT_DIR / "tmp_preprocessed"
RESULTS_PATH = OUTPUT_DIR / "results.json"
RESULTS_MD_PATH = OUTPUT_DIR / "results.md"
CONCLUSIONS_PATH = OUTPUT_DIR / "conclusions.md"

OUTPUT_DIR.mkdir(exist_ok=True)
TMP_DIR.mkdir(exist_ok=True)

info("Repositorio:", str(REPO_ROOT))
info("Benchmark:", str(BENCH_DIR))

[*] Repositorio: C:\Users\Ibai\Desktop\tfg github\manualito
[*] Benchmark: C:\Users\Ibai\Desktop\tfg github\manualito\docs\benchmarks\ocr_preprocessing


## 5. Dataset versionado

El manifest describe cada imagen y su papel dentro del benchmark. Los fragmentos de ground truth están en `dataset/ground_truth/*.txt`.


In [2]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8-sig")


def table(headers: list[str], table_rows: list[list[Any]]) -> str:
    values = [[str(value) for value in table_row] for table_row in table_rows]
    widths = [len(header) for header in headers]
    for value_row in values:
        for column_index, value in enumerate(value_row):
            widths[column_index] = max(widths[column_index], len(value))
    sep = " | "
    header = sep.join(
        headers[column_index].ljust(widths[column_index]) for column_index in range(len(headers))
    )
    rule = sep.join("-" * widths[column_index] for column_index in range(len(headers)))
    body = [
        sep.join(
            value_row[column_index].ljust(widths[column_index])
            for column_index in range(len(headers))
        )
        for value_row in values
    ]
    return "\n".join([header, rule, *body])


SYNTHETIC_FRAGMENTS = {
    "sintetico_manual_limpio": [
        "Preparación de la partida",
        "Cada jugador recibe cinco cartas y una ficha de ayuda.",
        "En tu turno puedes mover, robar una carta o resolver una acción.",
        "Gana quien complete tres objetivos antes de que termine la ronda.",
    ],
    "sintetico_manual_sombra": [
        "Preparación de la partida",
        "Cada jugador recibe cinco cartas y una ficha de ayuda.",
        "En tu turno puedes mover, robar una carta o resolver una acción.",
        "Gana quien complete tres objetivos antes de que termine la ronda.",
    ],
    "sintetico_manual_perspectiva": [
        "Preparación de la partida",
        "Cada jugador recibe cinco cartas y una ficha de ayuda.",
        "En tu turno puedes mover, robar una carta o resolver una acción.",
        "Gana quien complete tres objetivos antes de que termine la ronda.",
    ],
}


def load_font(size: int) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    for font_name in ["DejaVuSans.ttf", "Arial.ttf", "LiberationSans-Regular.ttf"]:
        try:
            return ImageFont.truetype(font_name, size)
        except OSError:
            continue
    return ImageFont.load_default()


def draw_synthetic_manual() -> Image.Image:
    image = Image.new("RGB", (1500, 2100), (238, 232, 218))
    draw = ImageDraw.Draw(image)
    draw.rectangle((110, 120, 1390, 1980), fill=(255, 253, 247), outline=(86, 72, 55), width=4)
    draw.rectangle((150, 165, 1350, 325), fill=(241, 229, 206), outline=(134, 96, 55), width=3)
    draw.text((190, 205), "Preparación de la partida", fill=(45, 38, 31), font=load_font(62))
    draw.text((192, 285), "Manual de prueba para OCR", fill=(90, 78, 66), font=load_font(32))

    body_font = load_font(38)
    small_font = load_font(31)
    y = 420
    paragraphs = [
        "1. Cada jugador recibe cinco cartas y una ficha de ayuda.",
        "2. En tu turno puedes mover, robar una carta o resolver una acción.",
        "3. Gana quien complete tres objetivos antes de que termine la ronda.",
    ]
    for paragraph in paragraphs:
        draw.text((190, y), paragraph, fill=(38, 35, 32), font=body_font)
        y += 86

    draw.line((190, y + 20, 1310, y + 20), fill=(118, 97, 72), width=3)
    y += 95
    table_rows = [
        ("Símbolo", "Efecto"),
        ("Sol", "Avanza una casilla adicional."),
        ("Llave", "Abre una puerta cerrada."),
        ("Carta", "Roba una carta de evento."),
    ]
    x1, x2 = 190, 570
    for row_index, (left, right) in enumerate(table_rows):
        row_y = y + row_index * 76
        fill = (248, 241, 229) if row_index == 0 else (255, 253, 247)
        draw.rectangle((x1, row_y, 1310, row_y + 76), fill=fill, outline=(155, 133, 105), width=2)
        draw.line((x2, row_y, x2, row_y + 76), fill=(155, 133, 105), width=2)
        draw.text((x1 + 24, row_y + 20), left, fill=(38, 35, 32), font=small_font)
        draw.text((x2 + 24, row_y + 20), right, fill=(38, 35, 32), font=small_font)

    note_y = y + len(table_rows) * 76 + 95
    draw.rounded_rectangle((190, note_y, 1310, note_y + 210), radius=18, fill=(247, 238, 217))
    note = (
        "Nota: si dos jugadores empatan, gana quien conserve más cartas. "
        "Si persiste el empate, comparten la victoria."
    )
    draw.multiline_text((230, note_y + 34), note, fill=(60, 51, 42), font=small_font, spacing=12)
    return image


def add_shadow_and_noise(image: Image.Image) -> Image.Image:
    arr = np.array(image).astype(np.int16)
    height, width = arr.shape[:2]
    shadow = np.linspace(5, 90, width, dtype=np.int16)
    arr -= np.tile(shadow, (height, 1))[..., None]
    rng = np.random.default_rng(20260605)
    arr += rng.normal(0, 6, arr.shape).astype(np.int16)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr).filter(ImageFilter.GaussianBlur(radius=0.55))


def add_perspective(image: Image.Image) -> Image.Image:
    arr = np.array(image)
    height, width = arr.shape[:2]
    src = np.float32([[0, 0], [width, 0], [width, height], [0, height]])
    dst = np.float32([[95, 30], [width - 70, 0], [width - 25, height - 80], [35, height - 25]])
    matrix = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(arr, matrix, (width, height), borderValue=(238, 232, 218))
    return Image.fromarray(warped)


def build_synthetic_image(case_id: str) -> Image.Image:
    image = draw_synthetic_manual()
    if case_id == "sintetico_manual_sombra":
        return add_shadow_and_noise(image)
    if case_id == "sintetico_manual_perspectiva":
        return add_perspective(image)
    return image


def ensure_synthetic_dataset(manifest_data: dict[str, Any]) -> None:
    if not INCLUDE_SYNTHETIC_IMAGES:
        manifest_data["images"] = [
            item for item in manifest_data["images"] if item["id"] not in SYNTHETIC_FRAGMENTS
        ]
        return

    GROUND_TRUTH_DIR.mkdir(exist_ok=True)
    for manifest_item in manifest_data["images"]:
        case_id = manifest_item["id"]
        if case_id not in SYNTHETIC_FRAGMENTS:
            continue
        image_path = DATASET_DIR / manifest_item["file"]
        gt_path = GROUND_TRUTH_DIR / f"{case_id}.txt"
        if FORCE_SYNTHETIC_IMAGES or not image_path.exists():
            build_synthetic_image(case_id).save(image_path)
        if FORCE_SYNTHETIC_IMAGES or not gt_path.exists():
            gt_path.write_text("\n".join(SYNTHETIC_FRAGMENTS[case_id]) + "\n", encoding="utf-8")


manifest = json.loads(read_text(DATASET_DIR / "manifest.json"))
ensure_synthetic_dataset(manifest)
cases: list[dict[str, Any]] = []

for manifest_item in manifest["images"]:
    image_path = DATASET_DIR / manifest_item["file"]
    gt_path = GROUND_TRUTH_DIR / f"{manifest_item['id']}.txt"
    gt_fragments = [line.strip() for line in read_text(gt_path).splitlines() if line.strip()]
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    cases.append(
        {**manifest_item, "image_path": image_path, "gt_path": gt_path, "fragments": gt_fragments}
    )

dataset_rows = [
    [
        benchmark_case["id"],
        benchmark_case["name"],
        benchmark_case["category"],
        benchmark_case.get("split", "-"),
        benchmark_case.get("difficulty", "-"),
        benchmark_case.get("rights", "-"),
        len(benchmark_case["fragments"]),
        benchmark_case["role"],
    ]
    for benchmark_case in cases
]
print(
    table(
        ["id", "imagen", "categoria", "split", "dificultad", "derechos", "GT", "papel"],
        dataset_rows,
    )
)


id                           | imagen                               | categoria            | split      | dificultad | derechos               | GT | papel                                                                  
---------------------------- | ------------------------------------ | -------------------- | ---------- | ---------- | ---------------------- | -- | -----------------------------------------------------------------------
periodico_granma             | Periódico Granma                     | prensa_real          | seed       | media      | CC BY-SA 4.0           | 5  | estrés: columnas, foto irregular y texto pequeño                       
manual_merck_veterinaria     | Manual Merck Veterinaria             | manual_real          | seed       | media      | CC0 1.0                | 6  | documento cercano al caso de uso: portada/manual con tipografías mixtas
diario_burgos_1928           | Diario de Burgos 1928                | historico            | seed       | alta      

## 6. Métricas

El ground truth actual está formado por fragmentos relevantes, no por una transcripción completa de la página. Para no penalizar texto adicional reconocido correctamente, cada fragmento se compara contra la mejor ventana aproximada del texto OCR.

Esto es adecuado para Manualito porque el texto acaba en un RAG: interesa que los fragmentos de reglas importantes aparezcan bien recuperados. Para una transcripción completa de página habría que añadir ground truth completo y evaluar también orden de lectura.


In [3]:
SPACE_RE = re.compile(r"\s+")
PUNCT_RE = re.compile(r"[^\w\sáéíóúüñÁÉÍÓÚÜÑ]+", re.UNICODE)


def normalize_text(text: str) -> str:
    text = text.replace("­", "")
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip().lower()


def levenshtein(a: Any, b: Any) -> int:
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(curr[j - 1] + 1, prev[j] + 1, prev[j - 1] + cost))
        prev = curr
    return prev[-1]


def cer(pred: str, gt: str) -> float:
    pred_norm = normalize_text(pred)
    gt_norm = normalize_text(gt)
    if not gt_norm:
        return 0.0 if not pred_norm else 1.0
    return min(levenshtein(pred_norm, gt_norm) / len(gt_norm), 1.0)


def wer(pred: str, gt: str) -> float:
    pred_words = normalize_text(pred).split()
    gt_words = normalize_text(gt).split()
    if not gt_words:
        return 0.0 if not pred_words else 1.0
    return min(levenshtein(pred_words, gt_words) / len(gt_words), 1.0)


def best_fragment_score(prediction: str, gt_fragment: str) -> dict[str, Any]:
    pred_words = normalize_text(prediction).split()
    gt_words = normalize_text(gt_fragment).split()
    if not pred_words:
        return {"cer": 1.0, "wer": 1.0, "match": ""}
    if not gt_words:
        return {"cer": 0.0, "wer": 0.0, "match": ""}

    min_len = max(1, len(gt_words) - 3)
    max_len = min(len(pred_words), len(gt_words) + 5)
    best = {"cer": 1.0, "wer": 1.0, "match": ""}
    best_rank = float("inf")

    for size in range(min_len, max_len + 1):
        for start in range(0, len(pred_words) - size + 1):
            candidate = " ".join(pred_words[start : start + size])
            c = cer(candidate, gt_fragment)
            w = wer(candidate, gt_fragment)
            rank = w + (0.25 * c)
            if rank < best_rank:
                best = {"cer": c, "wer": w, "match": candidate}
                best_rank = rank
    return best


def evaluate_prediction(prediction: str, fragments: list[str]) -> dict[str, Any]:
    scores = [best_fragment_score(prediction, gt_fragment) for gt_fragment in fragments]
    if not scores:
        return {"cer": None, "wer": None, "detected": 0, "fragment_scores": []}
    return {
        "cer": statistics.mean(score["cer"] for score in scores),
        "wer": statistics.mean(score["wer"] for score in scores),
        "detected": sum(score["wer"] <= DETECTED_WER_THRESHOLD for score in scores),
        "fragment_scores": scores,
    }


info("Métricas listas:", "CER, WER y matching por fragmentos")

[*] Métricas listas: CER, WER y matching por fragmentos


## 7. Técnicas evaluadas

Se prueban 28 configuraciones compactas de contraste, binarización, nitidez, reescalado, reducción de ruido, morfología y pipelines combinados.

La idea no es explorar cientos de combinaciones, sino tener suficientes parámetros para saber qué le conviene a cada motor sin convertir el anexo en una tabla interminable.


In [4]:
def gray_1ch(img: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img


def gray_bgr(img: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(gray_1ch(img), cv2.COLOR_GRAY2BGR)


def clahe_bgr(img: np.ndarray, clip: float = 2.0, tile: int = 8) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    return cv2.cvtColor(clahe.apply(gray_1ch(img)), cv2.COLOR_GRAY2BGR)


def otsu_bgr(img: np.ndarray) -> np.ndarray:
    _, out = cv2.threshold(gray_1ch(img), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.cvtColor(out, cv2.COLOR_GRAY2BGR)


def adaptive_bgr(img: np.ndarray, adaptive_method: int, block: int, c_value: int) -> np.ndarray:
    out = cv2.adaptiveThreshold(
        gray_1ch(img), 255, adaptive_method, cv2.THRESH_BINARY, block, c_value
    )
    return cv2.cvtColor(out, cv2.COLOR_GRAY2BGR)


def sharpen_bgr(img: np.ndarray, sigma: float = 2.0, amount: float = 1.0) -> np.ndarray:
    blur = cv2.GaussianBlur(img, (0, 0), sigma)
    return cv2.addWeighted(img, 1 + amount, blur, -amount, 0)


def resize_gray(img: np.ndarray, scale_factor: float) -> np.ndarray:
    resized = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_CUBIC)
    return gray_bgr(resized)


def resize_then(
    img: np.ndarray, scale_factor: float, operation: Callable[[np.ndarray], np.ndarray]
) -> np.ndarray:
    resized = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_CUBIC)
    return operation(resized)


def morph_bgr(img: np.ndarray, morph_op: int, kernel_size: int) -> np.ndarray:
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    out = cv2.morphologyEx(gray_1ch(img), morph_op, kernel)
    return cv2.cvtColor(out, cv2.COLOR_GRAY2BGR)


def bilateral_bgr(img: np.ndarray, diameter: int, sigma: int) -> np.ndarray:
    return cv2.bilateralFilter(img, diameter, sigma, sigma)


Technique = dict[str, Any]
TECHNIQUES: list[Technique] = []


def add_tech(
    id_: str, name: str, family: str, params: str, apply_fn: Callable[[np.ndarray], np.ndarray]
) -> None:
    TECHNIQUES.append(
        {"id": id_, "name": name, "family": family, "params": params, "apply": apply_fn}
    )


add_tech("baseline", "Sin preprocesado", "baseline", "original", lambda img: img.copy())
add_tech("gray", "Escala de grises", "color", "BGR2GRAY", gray_bgr)
add_tech("otsu", "Otsu", "binarización", "THRESH_OTSU", otsu_bgr)

if TECHNIQUE_PRESET == "extended":
    clahe_params = list(product([1.0, 2.0, 3.0, 4.0], [4, 8, 16]))
    adaptive_params = list(product([11, 21, 31], [2, 5, 10]))
    sharpen_params = list(product([1.0, 2.0, 3.0], [1.0, 1.5, 2.0]))
    resize_factors = [1.5, 2.0, 2.5, 3.0]
    bilateral_params = list(product([5, 9], [50, 75]))
else:
    clahe_params = [(1.5, 8), (2.0, 4), (2.0, 8), (3.0, 8)]
    adaptive_params = [(21, 3), (31, 5)]
    sharpen_params = [(1.0, 1.0), (2.0, 1.0), (2.0, 1.5), (3.0, 1.0)]
    resize_factors = [1.5, 2.0, 3.0]
    bilateral_params = [(5, 50), (9, 50)]

for clahe_clip, clahe_tile in clahe_params:
    add_tech(
        f"clahe_c{clahe_clip:g}_t{clahe_tile}",
        f"CLAHE c={clahe_clip:g} t={clahe_tile}",
        "contraste",
        f"clip={clahe_clip:g}, tile={clahe_tile}x{clahe_tile}",
        partial(clahe_bgr, clip=clahe_clip, tile=clahe_tile),
    )

for method_name, adaptive_method in [
    ("mean", cv2.ADAPTIVE_THRESH_MEAN_C),
    ("gaussian", cv2.ADAPTIVE_THRESH_GAUSSIAN_C),
]:
    for adaptive_block, adaptive_c in adaptive_params:
        add_tech(
            f"adaptive_{method_name}_b{adaptive_block}_c{adaptive_c}",
            f"Adaptativa {method_name} b={adaptive_block} c={adaptive_c}",
            "binarización",
            f"method={method_name}, block={adaptive_block}, C={adaptive_c}",
            partial(
                adaptive_bgr,
                adaptive_method=adaptive_method,
                block=adaptive_block,
                c_value=adaptive_c,
            ),
        )

for sharpen_sigma, sharpen_amount in sharpen_params:
    add_tech(
        f"sharpen_s{sharpen_sigma:g}_a{sharpen_amount:g}",
        f"Nitidez s={sharpen_sigma:g} a={sharpen_amount:g}",
        "nitidez",
        f"sigma={sharpen_sigma:g}, amount={sharpen_amount:g}",
        partial(sharpen_bgr, sigma=sharpen_sigma, amount=sharpen_amount),
    )

for resize_factor in resize_factors:
    add_tech(
        f"resize{resize_factor:g}_gray",
        f"Resize x{resize_factor:g} + grises",
        "reescalado",
        f"factor={resize_factor:g}, INTER_CUBIC",
        partial(resize_gray, scale_factor=resize_factor),
    )

for median_kernel in [3, 5]:
    add_tech(
        f"median_k{median_kernel}",
        f"Mediana k={median_kernel}",
        "ruido",
        f"ksize={median_kernel}",
        partial(cv2.medianBlur, ksize=median_kernel),
    )

for bilateral_diameter, bilateral_sigma in bilateral_params:
    add_tech(
        f"bilateral_d{bilateral_diameter}_s{bilateral_sigma}",
        f"Bilateral d={bilateral_diameter} s={bilateral_sigma}",
        "ruido",
        f"d={bilateral_diameter}, sigma={bilateral_sigma}",
        partial(bilateral_bgr, diameter=bilateral_diameter, sigma=bilateral_sigma),
    )

for morph_name, morph_op in [("open", cv2.MORPH_OPEN), ("close", cv2.MORPH_CLOSE)]:
    for morph_kernel in [2, 3]:
        add_tech(
            f"morph_{morph_name}_k{morph_kernel}",
            f"Morfológica {morph_name} k={morph_kernel}",
            "morfología",
            f"op={morph_name}, kernel={morph_kernel}x{morph_kernel}",
            partial(morph_bgr, morph_op=morph_op, kernel_size=morph_kernel),
        )

add_tech(
    "resize2_clahe",
    "Resize x2 + CLAHE",
    "pipeline",
    "factor=2, CLAHE c=2 t=8",
    partial(resize_then, scale_factor=2.0, operation=clahe_bgr),
)
add_tech(
    "resize2_otsu",
    "Resize x2 + Otsu",
    "pipeline",
    "factor=2, Otsu",
    partial(resize_then, scale_factor=2.0, operation=otsu_bgr),
)
if TECHNIQUE_PRESET == "extended":
    add_tech(
        "resize2_sharpen",
        "Resize x2 + nitidez",
        "pipeline",
        "factor=2, sigma=2, amount=1",
        partial(resize_then, scale_factor=2.0, operation=sharpen_bgr),
    )
    add_tech(
        "resize2_clahe_otsu",
        "Resize x2 + CLAHE + Otsu",
        "pipeline",
        "factor=2, CLAHE c=2 t=8, Otsu",
        lambda img: otsu_bgr(resize_then(img, 2.0, clahe_bgr)),
    )

technique_rows = [
    [technique["id"], technique["name"], technique["family"], technique["params"]]
    for technique in TECHNIQUES
]
print(table(["id", "técnica", "familia", "parámetros"], technique_rows))

id                        | técnica                       | familia      | parámetros                     
------------------------- | ----------------------------- | ------------ | -------------------------------
baseline                  | Sin preprocesado              | baseline     | original                       
gray                      | Escala de grises              | color        | BGR2GRAY                       
otsu                      | Otsu                          | binarización | THRESH_OTSU                    
clahe_c1_t4               | CLAHE c=1 t=4                 | contraste    | clip=1, tile=4x4               
clahe_c1_t8               | CLAHE c=1 t=8                 | contraste    | clip=1, tile=8x8               
clahe_c1_t16              | CLAHE c=1 t=16                | contraste    | clip=1, tile=16x16             
clahe_c2_t4               | CLAHE c=2 t=4                 | contraste    | clip=2, tile=4x4               
clahe_c2_t8               | CLAHE c=2

## 8. Motores OCR

Los motores se inicializan de forma perezosa. Por defecto el benchmark es estricto: si un motor pedido en `ENGINES` no se puede cargar o ejecutar, la celda falla para que no demos por buenos resultados incompletos.

Para ejecutar los tres de una, usa el entorno de Paddle GPU. El motor `paddle_cpu` usa el mismo paquete de Paddle, pero fuerza el dispositivo `cpu`.


In [5]:
@contextlib.contextmanager
def quiet_native_output():
    sys.stdout.flush()
    sys.stderr.flush()
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    saved_out = os.dup(1)
    saved_err = os.dup(2)
    py_out, py_err = sys.stdout, sys.stderr
    sys.stdout = io.StringIO()
    sys.stderr = io.StringIO()
    try:
        os.dup2(devnull_fd, 1)
        os.dup2(devnull_fd, 2)
        yield
    finally:
        sys.stdout.flush()
        sys.stderr.flush()
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(devnull_fd)
        os.close(saved_out)
        os.close(saved_err)
        sys.stdout = py_out
        sys.stderr = py_err


def paddle_gpu_available() -> bool:
    try:
        with quiet_native_output():
            paddle_module = importlib.import_module("paddle")
        return bool(
            paddle_module.is_compiled_with_cuda() and paddle_module.device.cuda.device_count() > 0
        )
    except (ImportError, RuntimeError, AttributeError, OSError):
        return False


def parse_paddle_result(result: Any) -> tuple[str, float | None]:
    texts: list[str] = []
    scores: list[float] = []
    for prediction_item in result or []:
        if isinstance(prediction_item, dict):
            rec_texts = prediction_item.get("rec_texts", [])
            rec_scores = prediction_item.get("rec_scores", [])
            texts.extend(str(text) for text in rec_texts)
            scores.extend(float(score) for score in rec_scores)
    confidence = statistics.mean(scores) if scores else None
    return " ".join(texts), confidence


def create_engine(requested_engine: str) -> dict[str, Any]:
    if requested_engine in {"paddle_cpu", "paddle_gpu"}:
        if requested_engine == "paddle_gpu" and not paddle_gpu_available():
            raise RuntimeError("Paddle GPU no disponible en este entorno")
        with quiet_native_output():
            paddleocr_module = importlib.import_module("paddleocr")
            paddle_ocr_class = paddleocr_module.PaddleOCR
            kwargs = {"use_textline_orientation": True, "lang": "es"}
            if requested_engine == "paddle_cpu":
                kwargs.update({"device": "cpu", "enable_mkldnn": False})
            else:
                kwargs.update({"device": "gpu"})
            ocr = paddle_ocr_class(**kwargs)

        def run(path: Path) -> dict[str, Any]:
            with quiet_native_output():
                text, confidence = parse_paddle_result(ocr.predict(str(path)))
            return {"text": text, "confidence": confidence}

        name = "PaddleOCR CPU" if requested_engine == "paddle_cpu" else "PaddleOCR GPU"
        return {"id": requested_engine, "name": name, "run": run}

    if requested_engine == "tesseract":
        import pytesseract
        from pytesseract import Output

        pytesseract.get_tesseract_version()

        def run(path: Path) -> dict[str, Any]:
            text = pytesseract.image_to_string(str(path), lang="spa", timeout=120)
            data = pytesseract.image_to_data(
                str(path), lang="spa", output_type=Output.DICT, timeout=120
            )
            confs = [
                int(conf)
                for conf in data.get("conf", [])
                if str(conf).lstrip("-").isdigit() and int(conf) > 0
            ]
            confidence = statistics.mean(confs) / 100 if confs else None
            return {"text": text.strip(), "confidence": confidence}

        return {"id": requested_engine, "name": "Tesseract", "run": run}

    raise ValueError(f"Motor no soportado: {requested_engine}")


info("Motores configurados:", ", ".join(ENGINES))

[*] Motores configurados: tesseract, paddle_cpu, paddle_gpu


## 9. Caché y ejecución

El hash de configuración cambia si cambian las imágenes, el manifest, el ground truth o la lista de técnicas. Si el hash cambia, la caché anterior se ignora salvo que se quiera comparar manualmente.


In [6]:
def digest_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def config_hash() -> str:
    digest = hashlib.sha256()
    digest.update(NOTEBOOK_VERSION.encode())
    digest.update(json.dumps(manifest.get("images", []), sort_keys=True).encode())
    digest.update(
        json.dumps(
            [
                {key: technique[key] for key in ("id", "name", "family", "params")}
                for technique in TECHNIQUES
            ],
            sort_keys=True,
        ).encode()
    )
    for benchmark_case in sorted(cases, key=lambda case_data: case_data["id"]):
        digest.update(benchmark_case["id"].encode())
        digest.update(digest_file(benchmark_case["image_path"]).encode())
        digest.update(digest_file(benchmark_case["gt_path"]).encode())
    return digest.hexdigest()


CONFIG_HASH = config_hash()


def empty_cache() -> dict[str, Any]:
    return {
        "notebook_version": NOTEBOOK_VERSION,
        "config_hash": CONFIG_HASH,
        "created_at": datetime.now(UTC).isoformat(),
        "engines": {},
    }


def load_cache() -> dict[str, Any]:
    if not RESULTS_PATH.exists():
        return empty_cache()
    cache_data = json.loads(read_text(RESULTS_PATH))
    if cache_data.get("config_hash") != CONFIG_HASH:
        warn("El dataset o la configuración han cambiado; se crea una caché nueva.")
        return empty_cache()
    return cache_data


def save_cache(cache_data: dict[str, Any]) -> None:
    RESULTS_PATH.write_text(json.dumps(cache_data, ensure_ascii=False, indent=2), encoding="utf-8")


def mean_field(rows: list[dict[str, Any]], field: str) -> float:
    return statistics.mean(row[field] for row in rows)


def summarize(summary_engine_id: str, result_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_tech: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for result_row in result_rows:
        by_tech[result_row["technique_id"]].append(result_row)

    baseline_rows_by_image: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for baseline_row in by_tech.get("baseline", []):
        baseline_rows_by_image[baseline_row["image_id"]].append(baseline_row)
    baseline_cer_by_image = {
        image_id: mean_field(baseline_rows, "cer")
        for image_id, baseline_rows in baseline_rows_by_image.items()
    }

    summary_rows = []
    for technique in TECHNIQUES:
        tech_rows = by_tech.get(technique["id"], [])
        if not tech_rows:
            continue
        deltas_by_image: dict[str, list[float]] = defaultdict(list)
        for tech_row in tech_rows:
            base_cer = baseline_cer_by_image.get(tech_row["image_id"])
            if base_cer is not None:
                deltas_by_image[tech_row["image_id"]].append(tech_row["cer"] - base_cer)

        deltas = [statistics.mean(values) for values in deltas_by_image.values()]
        summary_rows.append(
            {
                "engine_id": summary_engine_id,
                "technique_id": technique["id"],
                "technique_name": technique["name"],
                "family": technique["family"],
                "params": technique["params"],
                "cer": mean_field(tech_rows, "cer"),
                "wer": mean_field(tech_rows, "wer"),
                "delta_cer": statistics.mean(deltas) if deltas else 0.0,
                "improved_images": sum(delta < 0 for delta in deltas),
                "images": len({tech_row["image_id"] for tech_row in tech_rows}),
                "detected_fragments": sum(tech_row["detected"] for tech_row in tech_rows),
                "fragments": sum(tech_row["fragments"] for tech_row in tech_rows),
                "preprocess_s": mean_field(tech_rows, "preprocess_s"),
                "ocr_s": mean_field(tech_rows, "ocr_s"),
                "total_s": mean_field(tech_rows, "total_s"),
            }
        )
    return sorted(
        summary_rows,
        key=lambda summary_item: (
            summary_item["delta_cer"],
            summary_item["cer"],
            summary_item["total_s"],
        ),
    )


def run_engine(run_engine_id: str) -> dict[str, Any]:
    ocr_engine = create_engine(run_engine_id)
    info("Warm-up:", ocr_engine["name"])
    warm_img = cv2.imread(str(cases[0]["image_path"]))
    warm_path = TMP_DIR / f"warmup_{run_engine_id}.png"
    cv2.imwrite(str(warm_path), warm_img)
    ocr_engine["run"](warm_path)

    benchmark_rows: list[dict[str, Any]] = []
    for repeat_index in range(1, REPEATS + 1):
        for technique in TECHNIQUES:
            for benchmark_case in cases:
                source_img = cv2.imread(str(benchmark_case["image_path"]))
                if source_img is None:
                    raise RuntimeError(f"No se pudo leer {benchmark_case['image_path']}")

                pre_start = time.perf_counter()
                processed = technique["apply"](source_img)
                preprocess_s = time.perf_counter() - pre_start

                tmp_path = TMP_DIR / f"{run_engine_id}_{technique['id']}_{benchmark_case['id']}.png"
                cv2.imwrite(str(tmp_path), processed)

                ocr_start = time.perf_counter()
                output = ocr_engine["run"](tmp_path)
                ocr_s = time.perf_counter() - ocr_start

                metrics = evaluate_prediction(output["text"], benchmark_case["fragments"])
                benchmark_rows.append(
                    {
                        "engine_id": run_engine_id,
                        "engine_name": ocr_engine["name"],
                        "repeat": repeat_index,
                        "image_id": benchmark_case["id"],
                        "image_name": benchmark_case["name"],
                        "technique_id": technique["id"],
                        "technique_name": technique["name"],
                        "params": technique["params"],
                        "cer": metrics["cer"],
                        "wer": metrics["wer"],
                        "detected": metrics["detected"],
                        "fragments": len(benchmark_case["fragments"]),
                        "preprocess_s": preprocess_s,
                        "ocr_s": ocr_s,
                        "total_s": preprocess_s + ocr_s,
                        "confidence": output["confidence"],
                        "text": output["text"],
                    }
                )
            info(ocr_engine["name"], f"{technique['name']} listo")

    return {
        "status": "ok",
        "engine_id": run_engine_id,
        "engine_name": ocr_engine["name"],
        "run_at": datetime.now(UTC).isoformat(),
        "rows": benchmark_rows,
        "summary": summarize(run_engine_id, benchmark_rows),
    }


info("Hash de configuración:", CONFIG_HASH[:12])

[*] Hash de configuración: 451ce39f21ad


## 10. Informe

Esta celda imprime el resumen y escribe los `.md` para poder copiar conclusiones a la memoria.


In [7]:
def signed(value: float) -> str:
    return f"{value:+.4f}"


def markdown_table(headers: list[str], table_rows: list[list[Any]]) -> str:
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend("| " + " | ".join(str(value) for value in row) + " |" for row in table_rows)
    return "\n".join(lines)


def recommendation(summary_rows: list[dict[str, Any]]) -> dict[str, Any] | None:
    min_improved = max(1, len(cases) // 2 + len(cases) % 2)
    return next(
        (
            summary_item
            for summary_item in summary_rows
            if summary_item["technique_id"] != "baseline"
            and summary_item["delta_cer"] <= -0.005
            and summary_item["improved_images"] >= min_improved
        ),
        None,
    )


def summary_values(summary_item: dict[str, Any], with_family: bool = False) -> list[str]:
    values = [summary_item["technique_name"]]
    if with_family:
        values.append(summary_item["family"])
    return [
        *values,
        summary_item["params"],
        f"{summary_item['cer']:.4f}",
        signed(summary_item["delta_cer"]),
        f"{summary_item['wer']:.4f}",
        f"{summary_item['detected_fragments']}/{summary_item['fragments']}",
        f"{summary_item['preprocess_s']:.3f}s",
        f"{summary_item['ocr_s']:.3f}s",
    ]


def print_report(cache_data: dict[str, Any]) -> None:
    if not cache_data["engines"]:
        warn("No hay resultados todavía. Cambia RUN_BENCHMARK = True para ejecutar.")
        return

    for cached_engine_id, engine_result in cache_data["engines"].items():
        if engine_result.get("status") != "ok":
            warn(f"{cached_engine_id}: {engine_result.get('reason', 'skipped')}")
            continue
        print()
        ok(engine_result["engine_name"])
        report_rows = [summary_values(item) for item in engine_result["summary"][:TOP_N]]
        print(table(["técnica", "params", "CER", "dCER", "WER", "frag", "pre", "OCR"], report_rows))
        rec = recommendation(engine_result["summary"])
        if rec:
            ok(
                "Recomendación:",
                f"{rec['technique_name']} [{rec['params']}] ({signed(rec['delta_cer'])} CER)",
            )
        else:
            warn("No hay mejora suficientemente consistente frente a baseline.")


def write_reports(cache_data: dict[str, Any]) -> None:
    result_lines = ["# Resultados OCR/preprocesado", ""]
    conclusion_lines = ["# Conclusiones OCR/preprocesado", ""]

    for cached_engine_id, engine_result in cache_data["engines"].items():
        if engine_result.get("status") != "ok":
            conclusion_lines.append(
                f"- `{cached_engine_id}` queda pendiente: {engine_result.get('reason', 'skipped')}."
            )
            continue

        report_rows = [summary_values(item, with_family=True) for item in engine_result["summary"]]
        result_lines.extend([f"## {engine_result['engine_name']}", ""])
        result_lines.append(
            markdown_table(
                [
                    "Técnica",
                    "Familia",
                    "Parámetros",
                    "CER",
                    "dCER",
                    "WER",
                    "Fragmentos",
                    "Pre",
                    "OCR",
                ],
                report_rows,
            )
        )
        result_lines.append("")

        engine_name = engine_result["engine_name"]
        rec = recommendation(engine_result["summary"])
        if rec:
            tech_name = rec["technique_name"]
            tech_params = rec["params"]
            delta_cer = signed(rec["delta_cer"])
            conclusion_lines.append(
                f"- Para **{engine_name}**, la técnica recomendada es "
                f"**{tech_name}** (`{tech_params}`), con {delta_cer} CER frente a baseline."
            )
        else:
            conclusion_lines.append(
                f"- Para **{engine_name}**, no hay mejora consistente; "
                "mantener baseline o revisar más imágenes."
            )

    result_lines.append(f"_Generado: {datetime.now(UTC).isoformat()}_")
    RESULTS_MD_PATH.write_text("\n".join(result_lines), encoding="utf-8")
    CONCLUSIONS_PATH.write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
    info("Informes escritos:", f"{RESULTS_MD_PATH.name}, {CONCLUSIONS_PATH.name}")

## 11. Ejecutar o cargar resultados

Si `RUN_BENCHMARK = False`, esta celda solo carga resultados previos. Si no hay resultados, muestra un aviso y no hace nada destructivo.

Si `RUN_BENCHMARK = True` y `ALLOW_ENGINE_SKIP = False`, todos los motores definidos en `ENGINES` deben completarse. Si uno falla, la ejecución se detiene para no generar conclusiones parciales.


## 12. Fuentes y referencias del dataset

### Imágenes incluidas

| Fichero                        | Imagen                                 | Fuente                                                                                                                                                                                                                                                                                                               | Licencia/uso                    |
| ------------------------------ | -------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------- |
| `periodico_granma.jpg`         | Periódico *Granma*                     | [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Dos_ejemplares_del_Peri%C3%B3dico_Granma.jpg)                                                                                                                                                                                                            | CC BY-SA 4.0                    |
| `manual_merck_veterinaria.jpg` | *Manual Merck de Veterinaria*          | [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Manual-merck-de-veterinaria-cuarta-edicion-folleto-1994.jpg)                                                                                                                                                                                             | CC0 1.0                         |
| `diario_burgos_1928.jpg`       | *Diario de Burgos* (1928)              | [Biblioteca Virtual de Prensa Histórica](https://prensahistorica.mcu.es/en/publicaciones/verNumero.do?idNumero=1000474386)                                                                                                                                                                                           | CC BY 4.0 para la copia digital |
| `juego_mesa_palma_p1.jpg`      | *Juego de mesa Palma*                  | [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Juego_de_mesa_palma.pdf), renderizado desde PDF original                                                                                                                                                                                                 | CC BY-SA 4.0                    |
| `juego_oca_met.jpg`            | *Juego de la Oca*, MET                 | [Metropolitan Museum of Art](https://www.metmuseum.org/art/collection/search/717686)                                                                                                                                                                                                                                 | CC0 1.0                         |
| `juego_oca_loc.jpg`            | *Juego de la Oca*, Library of Congress | [Library of Congress](https://www.loc.gov/item/99615951/), convertido desde TIFF                                                                                                                                                                                                                                     | Public Domain Mark 1.0          |
| `croquet_manual_1865_p3.jpg`   | *How to play croquet* (1865)           | [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:How_to_play_croqu%C3%AAt_-_a_new_pocket_manual_of_complete_instructions_for_American_players,_illustrated_with_engravings_and_diagrams,_together_with_all_the_rules_of_the_game_(IA_howtoplaycroqu00adam).pdf), convertido desde JP2 de Internet Archive | Dominio público                 |

### Originales descargados

| Fichero                          | Uso                                                              |
| -------------------------------- | ---------------------------------------------------------------- |
| `raw/juego_mesa_palma.pdf`       | PDF original usado para renderizar `juego_mesa_palma_p1.jpg`.    |
| `raw/croquet_manual_1865.pdf`    | PDF original completo de Internet Archive.                       |
| `raw/croquet_manual_1865_p3.jp2` | Página original usada para generar `croquet_manual_1865_p3.jpg`. |
| `raw/rattfallan_rules_1819.pdf`  | Fuente revisada y no incluida en el benchmark evaluable.         |

### Imágenes sintéticas

Las imágenes sintéticas se generan de forma determinista al ejecutar el notebook si no existen todavía. Sirven como controles conocidos: texto limpio, texto con sombra/ruido y texto con perspectiva. No sustituyen a las fotos reales, pero ayudan a detectar si una técnica degrada casos sencillos.

### Criterio de uso

`juego_mesa_palma_p1.jpg` se marca como validación porque es el caso libre más cercano al uso real de Manualito: reglas modernas en español. Las imágenes históricas se marcan como `stress`; sirven para comprobar robustez ante documentos difíciles, pero no deberían decidir por sí solas el preprocesado por defecto.

Fuente revisada y no incluida: [Rattfallan board game 1819 rules](https://commons.wikimedia.org/wiki/File:R%C3%A5ttf%C3%A4llan_board_game_1819_rules.pdf). El original queda en `raw/`, pero no entra en el benchmark porque no es representativo para Manualito y mete una dificultad histórica/idiomática que desviaría la conclusión.

Para cada imagen se versiona su fichero, su entrada en `manifest.json`, su ground truth revisado y su referencia/licencia. Si una fuente futura no tiene licencia clara, queda fuera del dataset público.


In [8]:
results_cache = load_cache()
ENGINE_SKIP_ERRORS = (ImportError, RuntimeError, ValueError, OSError, AttributeError)

if RUN_BENCHMARK:
    for requested_engine_id in ENGINES:
        if (
            not FORCE
            and requested_engine_id in results_cache["engines"]
            and results_cache["engines"][requested_engine_id].get("status") == "ok"
        ):
            info("Caché reutilizada:", requested_engine_id)
            continue
        try:
            results_cache["engines"][requested_engine_id] = run_engine(requested_engine_id)
            save_cache(results_cache)
            ok("Resultado guardado:", requested_engine_id)
        except ENGINE_SKIP_ERRORS as exc:
            if ALLOW_ENGINE_SKIP:
                results_cache["engines"][requested_engine_id] = {
                    "status": "skipped",
                    "engine_id": requested_engine_id,
                    "reason": str(exc),
                }
                save_cache(results_cache)
                fail(f"{requested_engine_id}: {exc}")
                continue
            raise RuntimeError(
                f"{requested_engine_id} no se pudo ejecutar. "
                "El benchmark está en modo estricto porque ALLOW_ENGINE_SKIP = False."
            ) from exc
else:
    info("Modo lectura:", "RUN_BENCHMARK = False")

print_report(results_cache)

if results_cache["engines"]:
    write_reports(results_cache)


[*] Caché reutilizada: tesseract
[*] Caché reutilizada: paddle_cpu
[*] Warm-up: PaddleOCR GPU
[*] PaddleOCR GPU Sin preprocesado listo
[*] PaddleOCR GPU Escala de grises listo
[*] PaddleOCR GPU Otsu listo
[*] PaddleOCR GPU CLAHE c=1 t=4 listo
[*] PaddleOCR GPU CLAHE c=1 t=8 listo
[*] PaddleOCR GPU CLAHE c=1 t=16 listo
[*] PaddleOCR GPU CLAHE c=2 t=4 listo
[*] PaddleOCR GPU CLAHE c=2 t=8 listo
[*] PaddleOCR GPU CLAHE c=2 t=16 listo
[*] PaddleOCR GPU CLAHE c=3 t=4 listo
[*] PaddleOCR GPU CLAHE c=3 t=8 listo
[*] PaddleOCR GPU CLAHE c=3 t=16 listo
[*] PaddleOCR GPU CLAHE c=4 t=4 listo
[*] PaddleOCR GPU CLAHE c=4 t=8 listo
[*] PaddleOCR GPU CLAHE c=4 t=16 listo
[*] PaddleOCR GPU Adaptativa mean b=11 c=2 listo
[*] PaddleOCR GPU Adaptativa mean b=11 c=5 listo
[*] PaddleOCR GPU Adaptativa mean b=11 c=10 listo
[*] PaddleOCR GPU Adaptativa mean b=21 c=2 listo
[*] PaddleOCR GPU Adaptativa mean b=21 c=5 listo
[*] PaddleOCR GPU Adaptativa mean b=21 c=10 listo
[*] PaddleOCR GPU Adaptativa mean b=31 